# 🧠 Week 13 Lab — SOLUTIONS
## Artificial Neural Networks — Concepts

**Objectives:**
- Build a single unit (perceptron) from scratch and verify it matches logistic regression
- Construct an MLP and visualise how hidden layers create nonlinear decision boundaries
- Trace forward and backward passes through a small network with actual numbers
- Explore architecture choices (depth, width, activation functions)
- Compare MLP accuracy with all classical methods from Weeks 5–12
- Extract and visualise hidden representations
- Test MLP robustness to simulated electrode drift

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    cross_val_score, LeaveOneGroupOut, StratifiedKFold, train_test_split
)
from sklearn.metrics import accuracy_score, silhouette_score
from matplotlib.patches import Circle, Ellipse
from matplotlib.colors import ListedColormap

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})
cmap8 = plt.cm.get_cmap('Set1', 8)
skf = StratifiedKFold(5, shuffle=True, random_state=42)
logo = LeaveOneGroupOut()

## Upload Data

Upload `week8_data.pkl` — the same reaching dataset from Weeks 8–12.

This file contains:
- 480 trials (8 directions × 3 speeds × 20 subjects)
- 80 neural features (cosine-tuned M1 neurons, introduced in Week 6)
- 6 EMG features (muscle activations, introduced in Week 4)
- Subject labels for leave-one-subject-out cross-validation

**Note:** This week we return to **supervised learning** after Week 12's unsupervised methods.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload week8_data.pkl

In [ ]:
# Load data
with open('week8_data.pkl', 'rb') as f:
    D = pickle.load(f)

X_neural = D['neural_rates']   # (480, 80)
X_emg    = D['X_raw']          # (480, 6)
y_dir    = D['targets']        # 8 directions (0-7)
y_bin    = (D['labels'] == 'impaired').astype(int)
subjects = D['subjects']

sc_neural = StandardScaler().fit_transform(X_neural)
sc_emg = StandardScaler().fit_transform(X_emg)
X_pca2 = PCA(n_components=2).fit_transform(sc_neural)
dirs = np.unique(y_dir)

print(f'Dataset: {X_neural.shape[0]} trials, {X_neural.shape[1]} neurons, '
      f'{X_emg.shape[1]} muscles, {len(np.unique(subjects))} subjects')
print(f'Directions: {len(dirs)}, Healthy: {np.sum(y_bin==0)}, Impaired: {np.sum(y_bin==1)}')

---
## Part 1: The Building Block — A Single Unit 🟢

Before building a full neural network, let's verify that the simplest unit is something we already know: logistic regression from Week 5.

### Exercise 1.1: Perceptron from scratch

Implement the perceptron equation manually for the 0° vs 180° task (2 classes, 2D PCA space). Fit logistic regression, extract its weights, and compute the output by hand for a single trial.

**Expected result:** Your manual computation matches sklearn's prediction exactly.

In [ ]:
# Exercise 1.1: Perceptron = logistic regression
# Select 0° vs 180°
mask = np.isin(y_dir, [0, 4])
X_sub = X_pca2[mask]
y_sub = (y_dir[mask] == 4).astype(int)

# TODO: Fit LogisticRegression on X_sub, y_sub
# TODO: Extract weights (lr.coef_[0]) and bias (lr.intercept_[0])
# TODO: For the first trial, compute:
#   z = w1 * PC1 + w2 * PC2 + b
#   sigmoid = 1 / (1 + exp(-z))
# TODO: Verify your result matches lr.predict_proba()
# YOUR CODE HERE

### Exercise 1.2: Visualise the activation function

Plot the sigmoid curve and mark your computed values on it. Then **change the weights** and watch the prediction move along the curve.

**Try:** What happens when you double w₁? Flip its sign? Set both weights to zero?

In [ ]:
# Exercise 1.2: Sigmoid with your computed values
x_range = np.linspace(-6, 6, 200)
sig_curve = 1 / (1 + np.exp(-x_range))

# TODO: Plot the sigmoid curve
# TODO: Mark your computed z and sigmoid values as a red dot
# TODO: Annotate the regions: below 0.5 = predict 0°, above 0.5 = predict 180°
# YOUR CODE HERE

**Now explore:** Change the weights below and rerun to see how the prediction moves along the sigmoid curve. Each dot is a different set of weights applied to the same trial.

In [ ]:
# Exercise 1.2 (continued): Change weights, watch predictions change
trial = X_sub[0]  # same trial as before

# ═══ CHANGE THESE VALUES AND RERUN ═══
weight_experiments = [
    (w[0], w[1], b, 'Learned weights (original)'),
    (w[0]*2, w[1]*2, b, 'Double both weights'),
    (-w[0], -w[1], -b, 'Flip all signs'),
    (0.0, 0.0, 0.0, 'All zeros'),
    (w[0], 0.0, b, 'Only w₁ (ignore PC2)'),
]
# ═══════════════════════════════════════

# TODO: For each (w1, w2, b) in weight_experiments:
#   Compute z = w1*trial[0] + w2*trial[1] + b
#   Compute sigmoid = 1 / (1 + exp(-z))
#   Plot the point on the sigmoid curve
# YOUR CODE HERE

### Exercise 1.3: Decision boundary — when a line suffices

Plot the perceptron's decision boundary for 0° vs 180°.

**Expected result:** A single straight line perfectly separates the two groups (100% accuracy). This is logistic regression's boundary.

In [ ]:
# Exercise 1.3: Decision boundary for 0° vs 180°
# TODO: Create meshgrid over PCA space
# TODO: Plot LR decision boundary with contourf
# TODO: Overlay data points coloured by class
# YOUR CODE HERE

---
## Part 2: Hidden Layers and Nonlinear Boundaries 🟢

A single unit draws one straight line. What happens when we need nonlinear boundaries?

### Exercise 2.1: A problem LDA cannot solve

Group the 8 directions into **cardinal** (0°, 90°, 180°, 270°) vs **diagonal** (45°, 135°, 225°, 315°). These alternate around the ring — no straight line can separate them.

Compare LDA (linear) with MLP (nonlinear) on this task.

**Expected result:** LDA ≈ 50% (chance). MLP ≈ 95%. The hidden layer bends the boundary into shapes that linear methods cannot create.

In [ ]:
# Exercise 2.1: Cardinal vs diagonal
y_xor = np.array([int(d) % 2 for d in y_dir])  # 0=cardinal, 1=diagonal

# TODO: Evaluate LDA, LR, MLP(64), MLP(64,32) on this task (5-fold CV)
# TODO: Plot 1x3 decision boundaries in PCA space: LDA, MLP(64), MLP(64,32)
# YOUR CODE HERE

---
## Part 3: Backpropagation — Forward, Error, Backward 🟡

Trace the forward and backward passes through a trained network with actual numbers from our data.

### Exercise 3.1: Visualise forward and backward passes

Train a small MLP (2 inputs → 2 hidden → 1 output) on the 0° vs 180° task in PCA space. Extract the learned weights and trace one trial through the network step by step.

**What to look for:** In the forward pass, one hidden unit may be silent (relu outputs 0). In the backward pass, that silent unit receives no weight update.

In [ ]:
# Exercise 3.1: Forward and backward pass visualisation
# Train a tiny MLP(hidden_layer_sizes=(2,)) on 0° vs 180° in PCA space
mask = np.isin(y_dir, [0, 4])
X_sub = X_pca2[mask]
y_sub = (y_dir[mask] == 4).astype(int)

# TODO: Train MLPClassifier((2,), activation='relu') on X_sub, y_sub
# TODO: Extract weights: mlp.coefs_[0], mlp.intercepts_[0], etc.
# TODO: For trial 0, manually compute:
#   z1 = trial @ W1 + b1 (hidden pre-activation)
#   h1 = max(0, z1) (relu)
#   z2 = h1 @ W2 + b2 (output pre-activation)
#   sigmoid = 1 / (1 + exp(-z2))
# TODO: Visualise as a 3-panel figure (forward, error, backward)
# YOUR CODE HERE

---
## Part 4: Architecture and Training 🟡

Explore how architecture choices and training affect the MLP on our data.

### Exercise 4.1: Architecture sweep

Compare single-layer and multi-layer architectures on the standard 8-direction task.

**Expected result:** All architectures give ~94–95%. On 480 well-designed trials, architecture barely matters.

In [ ]:
# Exercise 4.1: Architecture sweep
archs = [(8,), (16,), (32,), (64,), (128,), (64,32), (128,64,32)]

# TODO: For each architecture, train MLP and compute 5-fold CV accuracy
# TODO: Plot as bar chart
# YOUR CODE HERE

### Exercise 4.2: Training curve — overfitting in action

Plot training vs test accuracy over epochs for the (64, 32) MLP.

**Expected result:** Training accuracy reaches 100% (memorisation). Test accuracy plateaus around 96%. The gap is overfitting.

In [ ]:
# Exercise 4.2: Training curve
X_tr, X_te, y_tr, y_te = train_test_split(
    sc_neural, y_dir, test_size=0.3, random_state=42, stratify=y_dir)

epochs = [1, 2, 5, 10, 20, 50, 100, 200, 500]

# TODO: For each epoch count, train MLP((64,32)) and record
#   training accuracy and test accuracy
# TODO: Plot both curves on semilog x-axis
# YOUR CODE HERE

### Exercise 4.3: Regularisation — does it matter here?

Sweep L2 regularisation strength (alpha) from 0.0001 to 1.0. Alpha is the hyperparameter that controls how heavily the network penalises large weights.

**Expected result:** Accuracy is flat across five orders of magnitude — regularisation barely matters on 480 clean trials. The bottleneck is data size, not model flexibility.

In [ ]:
# Exercise 4.3: Regularisation (alpha sweep)
alphas = [0.0001, 0.001, 0.01, 0.1, 1.0]

# TODO: For each alpha, train MLP((64,32), alpha=alpha) and compute 5-fold CV accuracy
# TODO: Plot accuracy vs alpha on semilog x-axis
# YOUR CODE HERE

---
## Part 5: Hidden Representations 🟡

The MLP's hidden layer produces a 32D learned feature space. How does it compare with PCA and LDA?

**Key distinction:** Accuracy measures prediction correctness on held-out data. Silhouette measures how well-separated the clusters are in feature space *before* classification. Two methods can have similar accuracy but very different silhouette scores.

### Exercise 5.1: Compare feature spaces

Extract the MLP's hidden representations and compare with PCA and LDA using silhouette scores.

**Expected result:** MLP hidden space has higher silhouette (~0.62) than PCA (~0.44) or LDA (~0.46), meaning directions are more cleanly separated — even though classification accuracy is similar.

In [ ]:
# Exercise 5.1: Hidden representations compared
mlp = MLPClassifier((64,32), max_iter=300, random_state=42).fit(sc_neural, y_dir)

# TODO: Extract hidden layer activations (apply weights + relu manually)
# TODO: Compute silhouette scores for PCA, LDA, and MLP hidden space
# TODO: Visualise all three as 2D scatter plots coloured by direction
# YOUR CODE HERE

---
## Part 6: Full Comparison and Drift Robustness 🔴

Add the MLP to the complete method comparison and test drift robustness.

### Exercise 6.1: 10-method comparison

Evaluate all 10 methods on the neural 8-direction task (LOSO).

**Expected result:** MLP matches classical methods (~94%) without beating them. But it is the only method that learns its own features.

In [ ]:
# Exercise 6.1: 10-method comparison (LOSO)
methods = {
    'NB': GaussianNB(),
    'LR': LogisticRegression(C=10, max_iter=2000),
    'KNN': KNeighborsClassifier(5),
    'Lin SVM': SVC(kernel='linear', C=1),
    'RBF SVM': SVC(kernel='rbf', C=10, gamma='scale'),
    'LDA': LinearDiscriminantAnalysis(),
    'RF': RandomForestClassifier(200, random_state=42),
    'MLP': MLPClassifier((64,32), max_iter=300, random_state=42),
}

# TODO: Evaluate all methods with LOSO
# TODO: Plot bar chart
# YOUR CODE HERE

### Exercise 6.2: Drift robustness

Add Gaussian noise (σ = 0, 1, 2, 5, 10) to the neural features and compare how MLP degrades vs classical methods.

**Expected result:** MLP degrades more slowly than RF at moderate drift (σ=5) because its hidden representations combine many neurons — if one drifts, others compensate.

In [ ]:
# Exercise 6.2: Drift robustness
drift_levels = [0, 1, 2, 5, 10]

# TODO: For each sigma, add noise to X_neural
# TODO: Evaluate LDA, LR, RF, MLP at each noise level (5-fold CV)
# TODO: Plot accuracy vs drift for all methods
# YOUR CODE HERE

---
## 💭 Thought Exercise

1. In Exercise 1.1, you verified that a single unit is logistic regression. If that's the case, why do we need hidden layers? What specific limitation did Exercise 2.1 expose?

2. The MLP achieves ~94% on 8-direction decoding — slightly *lower* than LDA (95.4%). Why doesn't the more complex model win? What would need to change about the data for the MLP to show its advantage?

3. In Exercise 5.1, the MLP's hidden space had a higher silhouette score than LDA but similar accuracy. Explain why a richer representation doesn't always translate to better classification.

4. The drift robustness in Exercise 6.2 showed the MLP holding up better than RF at σ=5. Explain *why* in terms of how the hidden layer combines neurons.

5. A colleague says: "Neural networks are always better than classical methods." Based on this week's results, how would you respond?